In [18]:
from Bio import SeqIO
import choppy as cp
import primer3
from collections import defaultdict
import re
from functools import lru_cache
import pickle
from pathlib import Path
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
import os
from Bio.Seq import Seq

In [19]:
all_sequences = list(SeqIO.parse("data/20240414-forSveta.fa", "fasta"))

# seq_names = ["Maizel_COS-AT1G27430-GYF2", "Maizel_COS-SETH5", 
#              "Pereira_COS-SynDNA-f1", "Pereira_COS-SynDNA-f2",
#              "PV252688r_p6utr", "PQ537341r_p6utr,"
#              "PQ488556r_p6utr", "PX021458r_p6utr",
#              "MZ289137_rep", "OR500095r_naive"]

# sequences = [seq for seq in all_sequences if seq.id in seq_names]

sequences = all_sequences

for seq in sequences:
    seq.seq = seq.seq.upper()
    
# I just haven't decided whether I want a list or a dict
sequences_by_id = {seq.id: seq for seq in sequences}

In [20]:

CONFIG = {
    'kmer_size': 15,
    'max_frag_length': 1000,
    'min_frag_length': 200,
    'min_overlap': 50,
    'max_overlap': 100,
    # The model will end the segment after it reaches this length
    # It is also penalized for not reaching it, though it is possible to end prematurely
    'opt_segment_length': 5000, 
    'min_segment_length': 1000,
    # Space for TT1
    'segment_offset_left': 86,
    'segment_offset_right': 116,
    # Space for TT2
    'seq_offset_left': 126,
    'seq_offset_right': 105,
    # Optimal primer length, the range around it is generally allowed
    'min_primer_length': 17,
    'max_primer_length': 30,
    # Used to penalize primers deviating from the optimal length
    'opt_primer_len': 21,
    'max_primer_len_diff': 3,
    # Use to search for mispriming, 
    # misprime Tm is calculated only for matching kmer at 3' end of primer
    'primer_3prime_anchor': 6,
    'max_misprime_tm': 47.0,
    # If overlaps don't differ much, that is the distance between them
    # If there is considerable difference in neighbourhood, this parameter is ignored
    'min_step': 10,
    # Standard primer3 parameters
    'min_gc': 0.3,
    'max_gc': 0.7,
    'min_tm': 59.0,
    'max_tm': 65.0,
    'max_hairpin_tm': 26.0,
    'max_homodimer_tm': 47.0,
    'max_heterodimer_tm': 47.0,
    'max_3_self_tm': 37.0,
    # params to match NEB Phusion HF Tm calculator
    'tm_params': dict(
        mv_conc=222.0, dv_conc=0.0, dntp_conc=0.0, dna_conc=500.0,
        tm_method='santalucia', salt_corrections_method='schildkraut',
    ),
    'poly_x_pattern': re.compile(r'(A{5,}|T{5,}|G{5,}|C{5,})'),
    # Due to technical reasons, just having a regexp for CGclamp is not enough
    'clamp_length': 3,
    'clamp_pattern': re.compile(r'[GC][AT][GC]|[AT][GC][GC]'),
    # This is the pattern for primers' 5' end. Currenntly allows anything.
    'five_prime_clamp_pattern': re.compile(r'^[ATGC]'),
    'seg_boundary_pattern': re.compile(r'GC')
}

In [21]:
bg_trie = cp.load_trie("../data/S_cerevisiae-R64-GCA_000146045_cat_15.marisa")
seq_tries = {}
for seq in sequences:
    trie = cp.create_kmer_trie(seq, CONFIG['kmer_size'], bg=False)
    seq_tries[seq.id] = trie 

bg_regions = {}
cur_seq_regions = {}
for seq in sequences:
    bg_regions[seq.id] = cp.find_non_homologous_regions(seq, bg_trie, [], 
                                                        CONFIG['kmer_size'], 
                                                        threshold=CONFIG['min_overlap'])
    cur_seq_regions[seq.id] = cp.find_non_homologous_regions(seq, seq_tries[seq.id], bg_trie, 
                                                             CONFIG['kmer_size'], 
                                                             threshold=CONFIG['min_overlap'])

Processing sequence: 100%|██████████| 5911/5911 [00:00<00:00, 1241675.31it/s]


Found 18 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Processing sequence: 100%|██████████| 8495/8495 [00:00<00:00, 1742583.87it/s]


Found 0 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Processing sequence: 100%|██████████| 21124/21124 [00:00<00:00, 2481806.10it/s]


...

Processing sequence: 100%|██████████| 8137/8137 [00:00<00:00, 1463258.95it/s]


Found 2 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Processing sequence: 100%|██████████| 8096/8096 [00:00<00:00, 1467779.78it/s]


Found 4 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Finding non-homologous regions: 100%|██████████| 8096/8096 [00:00<00:00, 790987.31it/s]


## Functions for primer search

(And for local neighborhood construction)

In [22]:
def reverse_complement(seq):
    return seq[::-1].translate(str.maketrans('ATGC', 'TACG'))

# helper search utilities (assume lists are sorted ascending)
def next_val(a, val, default=None):
    return next((x for x in a if x > val), default)

def prev_val(a, val, default=None):
    return next((x for x in reversed(a) if x < val), default)

def compute_homfree_ranges(seq_str, kmer_size):
    """Return a list of (start,end) homfree ranges for each base in seq_str.

    A position i is annotated with the nearest previous/next k-mer collision
    adjusted to k-mer coordinates, matching the original inline logic.
    """
    kmers = defaultdict(list)
    for i in range(len(seq_str) - kmer_size + 1):
        kmer = seq_str[i:i+kmer_size]
        kmers[kmer].append(i)
        kmers[reverse_complement(kmer)].append(i)

    kmers = {k: v for k, v in kmers.items() if len(v) > 1}

    homfree_ranges = [(0, len(seq_str))] * len(seq_str)

    for i in range(len(seq_str)):
        start, end = homfree_ranges[i]
        if i < len(seq_str) - kmer_size + 1:
            kmer = seq_str[i:i+kmer_size]
            if kmer in kmers:
                n_val = next_val(kmers[kmer], i)
                if n_val is not None:
                    end = min(end, n_val + kmer_size - 1)
        if i >= kmer_size - 1:
            kmer = seq_str[i-kmer_size+1:i+1]
            if kmer in kmers:
                p_val = prev_val(kmers[kmer], i - kmer_size + 1)
                if p_val is not None:
                    start = max(start, p_val + 1)
        homfree_ranges[i] = (start, end)

    return homfree_ranges

def build_anchor_kmers(sequences, anchor_len, reverse_position = True):
    """Builds the 3' k-mer hash map for fast off-target screening."""
    anchor_kmers = defaultdict(list)
    for seq in sequences:
        seq_str = str(seq.seq).upper()
        for pos in range(len(seq_str) - anchor_len + 1):
            kmer_fwd = seq_str[pos:pos + anchor_len]
            kmer_rev = reverse_complement(kmer_fwd)
            
            anchor_kmers[kmer_fwd].append((seq.id, pos + anchor_len - 1, "forward"))
            if reverse_position:
                anchor_kmers[kmer_rev].append((seq.id, len(seq_str) - pos - anchor_len, "reverse"))
            else:
                anchor_kmers[kmer_rev].append((seq.id, pos + anchor_len - 1, "reverse"))
            
    return anchor_kmers

def check_misprime(primer_cand, native_3_prime_pos, side, seq_id, anchor_kmers, sequences_by_id, cfg):
    """Validates the primer against the k-mer map to ensure no high-Tm off-targets."""
    anchor_3 = primer_cand[-cfg['primer_3prime_anchor']:]
    
    for anchor_seq_id, anchor_pos, anchor_side in anchor_kmers.get(anchor_3, []):
        # Allow binding to the intended on-target site
        if anchor_side == side and anchor_seq_id == seq_id and anchor_pos == native_3_prime_pos:
            continue
            
        anchor_seq = str(sequences_by_id[anchor_seq_id].seq).upper()
        
        # Extract the off-target sequence depending on strand orientation
        if anchor_side == "forward":
            pos_misprime = anchor_seq[max(0, anchor_pos - cfg['max_primer_length'] + 1):anchor_pos + 1]
        else:
            pos_misprime = reverse_complement(anchor_seq[anchor_pos:min(len(anchor_seq), anchor_pos + cfg['max_primer_length'])])
            
        if primer3.calc_heterodimer_tm(primer_cand, reverse_complement(pos_misprime)) > cfg['max_misprime_tm']:
            return True
            
    return False

def check_local_misprime(primer_cand, seq_str, native_3_prime_pos, side, seq_id, anchor_kmers, cfg):
    """Checks for potential mispriming within the same potential fragment."""
    anchor_3 = primer_cand[-cfg['primer_3prime_anchor']:]
    
    for anchor_seq_id, anchor_pos, anchor_side in anchor_kmers.get(anchor_3, []):
        if anchor_seq_id != seq_id:
            continue
        if anchor_side != side:
            continue
        if anchor_pos == native_3_prime_pos:
            continue
        if anchor_pos - native_3_prime_pos > cfg['max_frag_length'] or native_3_prime_pos > anchor_pos:
            continue

        pos_misprime = seq_str[max(0, anchor_pos - cfg['max_primer_length'] + 1):anchor_pos + 1]
            
        if primer3.calc_heterodimer_tm(primer_cand, reverse_complement(pos_misprime)) > cfg['max_misprime_tm']:
            return True            
    return False

def find_primers_in_regions(seq_record, regions, side, anchor_kmers, sequences_by_id, cfg):
    """Finds primer candidates for a given sequence, regions, and orientation."""
    primer_candidates = []
    seq_id = seq_record.id
    original_seq = str(seq_record.seq).upper()
    
    if side == "forward":
        search_seq = original_seq
        search_regions = regions
    elif side == "reverse":
        search_seq = reverse_complement(original_seq)
        seq_len = len(original_seq)
        search_regions = [(seq_len - r[1], seq_len - r[0]) for r in regions]
    else:
        raise ValueError("Side must be 'forward' or 'reverse'")

    for region in search_regions:
        clamp_matches = cfg['clamp_pattern'].finditer(search_seq, region[0], region[1])
        
        for m in clamp_matches:
            range_start = max(m.start() - (cfg['max_primer_length'] - cfg['clamp_length']), region[0])
            range_end = m.start() - cfg['min_primer_length'] + cfg['clamp_length'] + 1
            
            # Extend 5' -> 3'
            for pr_start in range(range_end - 1, range_start - 1, -1):
                primer_cand = search_seq[pr_start:m.end()]
                if cfg['poly_x_pattern'].search(primer_cand): break
                if not cfg['five_prime_clamp_pattern'].search(primer_cand): continue
                
                gc_content = (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand)
                if gc_content < cfg['min_gc'] or gc_content > cfg['max_gc']: continue
                
                tm = primer3.calc_tm(primer_cand, **cfg['tm_params'])
                if tm > cfg['max_tm']: break
                if tm < cfg['min_tm']: continue
                if primer3.calc_hairpin_tm(primer_cand) > cfg['max_hairpin_tm']: continue
                if primer3.calc_homodimer_tm(primer_cand) > cfg['max_homodimer_tm']: continue
                if primer3.calc_end_stability(primer_cand, primer_cand).tm > cfg['max_3_self_tm']: continue
                
                if side == "forward":
                    native_3_prime_pos = m.end() - 1
                    pos_tuple = (pr_start, m.end())
                else:
                    native_3_prime_pos = len(original_seq) - m.end()
                    pos_tuple = (len(original_seq) - m.end(), len(original_seq) - pr_start)
                
                if check_local_misprime(primer_cand, search_seq, native_3_prime_pos, side, seq_id, anchor_kmers, cfg):
                    break
                    
                primer_candidates.append({
                    'seq': primer_cand,
                    'gc_content': gc_content,
                    'tm': tm,
                    'pos': pos_tuple,
                    'side': side
                })
                
    return primer_candidates

def find_primer_flanked_overlaps(seq_record, primer_regions, overlap_regions, anchor_kmers, sequences_by_id, cfg):
    """Finds all valid primer pairs that flank overlaps within the specified regions."""
    forward_primers = find_primers_in_regions(seq_record, primer_regions, "forward", anchor_kmers, sequences_by_id, cfg)
    reverse_primers = find_primers_in_regions(seq_record, primer_regions, "reverse", anchor_kmers, sequences_by_id, cfg)
    
    primer_flanked_overlaps = []

    for reg in overlap_regions:
        reg_forward_primers = list(filter(lambda x: x['pos'][0] >= reg[0] and x['pos'][1] <= reg[1], forward_primers))
        reg_reverse_primers = list(filter(lambda x: x['pos'][0] >= reg[0] and x['pos'][1] <= reg[1], reverse_primers))
        if len(reg_forward_primers) > 0 and len(reg_reverse_primers) > 0:
            for fwd in reg_forward_primers:
                for rev in reg_reverse_primers:
                    overlap_start = fwd['pos'][0]
                    overlap_end = rev['pos'][1]
                    if overlap_end - overlap_start >= cfg['min_overlap'] and overlap_end - overlap_start <= cfg['max_overlap']:
                        primer_flanked_overlaps.append({
                            'forward': fwd,
                            'reverse': rev,
                            'pos': (overlap_start, overlap_end)
                        })
    return primer_flanked_overlaps

def get_relaxed_overlaps(seq_id, gap_pos, anchor_kmers, sequences_by_id, cfg, extra_weight, **cfg_overrides):
    """Generate primer-flanked overlaps for hard-to-bridge gaps using relaxed parameters.

    Args:
        seq_id: sequence identifier
        gap_pos: a single (start, end) tuple or a list of (start, end) tuples
                 defining the gap region(s) to search within
        anchor_kmers: pre-built anchor k-mer map (from build_anchor_kmers)
        sequences_by_id: dict mapping seq_id -> SeqRecord
        cfg: base configuration dict; will not be mutated
        extra_weight: numeric value assigned to the 'extra_weight' field of every
                      returned overlap (used downstream by the shortest-path cost)
        **cfg_overrides: any config keys to override, e.g. max_tm=67, min_tm=57

    Returns:
        list of overlap dicts (same schema as find_primer_flanked_overlaps output)
        each extended with an 'extra_weight' key.
    """
    relaxed_cfg = {**cfg, **cfg_overrides}
    seq_record = sequences_by_id[seq_id]
    regions = gap_pos if isinstance(gap_pos, list) else [gap_pos]
    overlaps = find_primer_flanked_overlaps(
        seq_record, regions, regions, anchor_kmers, sequences_by_id, relaxed_cfg
    )
    for ov in overlaps:
        ov['extra_weight'] = extra_weight
    return overlaps


Running the abovedefined functions to get primer-flanked overlaps. They sit inside homology free regions relative to the background. The sequence-related homology is taken into account later.

In [23]:
anchor_kmers = build_anchor_kmers(sequences, CONFIG['primer_3prime_anchor'])

primer_flanked_overlaps = {}

for seq in sequences:
    print(seq.id)
    primer_flanked_overlaps[seq.id] = find_primer_flanked_overlaps(seq, bg_regions[seq.id], bg_regions[seq.id], anchor_kmers, sequences_by_id, CONFIG)


Maizel_COS-AT1G27430-GYF2
Maizel_COS-SETH5
Pereira_COS-SynDNA-f1
Pereira_COS-SynDNA-f2
PV252688r_p6utr
MK050105r_p6utr
MN450855r_p6utr
PQ488560r_p6utr
LC177792r_p6utr
AB890001r_p6utr
MH184583_rep
MG020022r_naive
OR500095r_p6utr
MN450853r_p6utr
JN998607r_p6utr
PQ537341r_p6utr
MZ289137_rep
MZ542728r_p6utr
PQ541186r_p6utr
ON644869r_p6utr
MG020022r_p6utr
MT840363_rep
OP610066r_p6utr
OR500095r_naive
PX021458r_p6utr
PQ488556r_p6utr


Now, getting local neighborhood ranges and constructing the final fragment overlaps

In [24]:
homfree_ranges = {}
for seq in sequences:
    seq_str = str(seq.seq).upper()
    homfree_ranges[seq.id] = compute_homfree_ranges(seq_str, CONFIG['kmer_size'])

def get_overlap_range(start, end, homfree_ranges, kmer_size):
    ov_end = min([el[1] for el in homfree_ranges[start:end - kmer_size + 1]])
    ov_start = max([el[0] for el in homfree_ranges[start + kmer_size - 1:end]])
    return ov_start, ov_end

for seq in sequences:
    seq_id = seq.id
    for ov in primer_flanked_overlaps[seq_id]:
        ov["range"] = get_overlap_range(ov["pos"][0], ov["pos"][1], homfree_ranges[seq_id], CONFIG['kmer_size'])
    primer_flanked_overlaps[seq_id] = [
        ov
        for ov in primer_flanked_overlaps[seq_id]
        if not (
            ov["range"][0] > ov["pos"][1] - CONFIG["min_frag_length"]
            or ov["range"][1] < ov["pos"][0] + CONFIG["min_frag_length"]
        )
    ]

In [25]:
# to bridge the complex repetitive gap we add two sort of primer-flanked, shorter overlaps
# only one of them is expected to be used, but let algorithm decide, which one
seq_id = 'Pereira_COS-SynDNA-f2'

pos = (16411, 16411 + 43)
seq_str = str(sequences_by_id[seq_id].seq)
print(seq_str[pos[0]:pos[1]])
primer_cand = "GTAGTCACATACCTGAAGAGGCAC"
print("forward one:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
# Well, that's a very nasty hairpin, to be honest...
forward_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[0], pos[0] + len(primer_cand)),
                "side": "forward"}

primer_cand = reverse_complement("GGCAGAAAGTTTCACCTGTTCT")
print("reverse one:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
reverse_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[1] - len(primer_cand), pos[1]),
                "side": "reverse"}
range_ov = get_overlap_range(pos[0], pos[1], homfree_ranges[seq_id], CONFIG['kmer_size'])
print(f"Overlap range: {range_ov}")
primer_flanked_overlaps[seq_id].append({
    "forward": forward_pr,
    "reverse": reverse_pr,
    "pos": (pos[0], pos[1]),
    "range": range_ov
})

pos = (16928, 16928 + 40)
seq_str = str(sequences_by_id[seq_id].seq)
print(seq_str[pos[0]:pos[1]])
primer_cand = "TTACTCACAACATACAGAGAAGCC"
print("forward two:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
# Well, that's a very nasty hairpin, to be honest...
forward_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[0], pos[0] + len(primer_cand)),
                "side": "forward"}

primer_cand = reverse_complement("GAGAAGCCGAGTATTTTCTACCAA")
print("reverse two:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
reverse_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[1] - len(primer_cand), pos[1]),
                "side": "reverse"}
range_ov = get_overlap_range(pos[0], pos[1], homfree_ranges[seq_id], CONFIG['kmer_size'])
print(f"Overlap range: {range_ov}")
primer_flanked_overlaps[seq_id].append({
    "forward": forward_pr,
    "reverse": reverse_pr,
    "pos": (pos[0], pos[1]),
    "range": range_ov
})

GTAGTCACATACCTGAAGAGGCACAGAAAGTTTCACCTGTTCT
forward one:
62.121293968297095
44.32879191090444
-46.22747264652193
-42.49151514903241
reverse one:
60.740541141579286
36.73831186622289
3.2172850196243985
5.577948619868437
Overlap range: (0, 18697)
TTACTCACAACATACAGAGAAGCCGAGTATTTTCTACCAA
forward two:
59.99377007478091
32.593277392564346
-20.710488676262344
-126.87021508389176
reverse two:
60.04004866748613
30.896312429318584
-42.90228704672958
-101.04725393346138
Overlap range: (0, 18082)


## Check for hard-to-bridge gaps

In [26]:
gap_threshold = 1.0 * CONFIG['max_frag_length'] - 2 * CONFIG['max_overlap']
gaps = {}

for seq in sequences:
    seq_len = len(seq.seq)
    gaps[seq.id] = []
    
    if primer_flanked_overlaps[seq.id]:
        overlaps = sorted([
            ov['pos'] 
            for ov in primer_flanked_overlaps[seq.id]
        ])
        
        if overlaps[0][0] > gap_threshold:
            gaps[seq.id].append((0, overlaps[0][0]))
        
        for i in range(len(overlaps) - 1):
            gap_start = overlaps[i][1]
            gap_end = overlaps[i + 1][0]
            gap_size = gap_end - gap_start
            if gap_size > gap_threshold:
                gaps[seq.id].append((gap_start, gap_end))
        
        if overlaps[-1][1] < seq_len - gap_threshold:
            gaps[seq.id].append((overlaps[-1][1], seq_len))
    else:
        gaps[seq.id].append((0, seq_len))

for seq in sequences:
    print(f"\n{seq.id}: {len(gaps[seq.id])} gaps found")
    for i, (gap_start, gap_end) in enumerate(gaps[seq.id]):
        print(f"  Gap {i+1}: {gap_start}-{gap_end} (length: {gap_end - gap_start})")


Maizel_COS-AT1G27430-GYF2: 1 gaps found
  Gap 1: 2383-3388 (length: 1005)

Maizel_COS-SETH5: 1 gaps found
  Gap 1: 3425-4355 (length: 930)

Pereira_COS-SynDNA-f1: 1 gaps found
  Gap 1: 19741-20638 (length: 897)

Pereira_COS-SynDNA-f2: 1 gaps found
  Gap 1: 22695-23546 (length: 851)

PV252688r_p6utr: 0 gaps found

MK050105r_p6utr: 0 gaps found

MN450855r_p6utr: 0 gaps found

PQ488560r_p6utr: 0 gaps found

LC177792r_p6utr: 0 gaps found

AB890001r_p6utr: 0 gaps found

MH184583_rep: 0 gaps found

MG020022r_naive: 0 gaps found

OR500095r_p6utr: 0 gaps found

MN450853r_p6utr: 0 gaps found

JN998607r_p6utr: 0 gaps found

PQ537341r_p6utr: 0 gaps found

MZ289137_rep: 0 gaps found

MZ542728r_p6utr: 0 gaps found

PQ541186r_p6utr: 0 gaps found

ON644869r_p6utr: 0 gaps found

MG020022r_p6utr: 0 gaps found

MT840363_rep: 0 gaps found

OP610066r_p6utr: 0 gaps found

OR500095r_naive: 0 gaps found

PX021458r_p6utr: 0 gaps found

PQ488556r_p6utr: 0 gaps found


In [27]:
for seq_id in gaps:
    if len(gaps[seq_id]) > 0:
        print(f"attempting to bridge gaps in {seq_id}")
        regions = cp.get_region_intersects(gaps[seq_id], bg_regions[seq_id], threshold=40)
        relaxed_overlaps = get_relaxed_overlaps(seq_id, regions, anchor_kmers, sequences_by_id, CONFIG, extra_weight=200.0, max_tm=67.0, min_tm=57.0, min_overlap=40)
        
        for ov in relaxed_overlaps:
            ov["range"] = get_overlap_range(ov["pos"][0], ov["pos"][1], homfree_ranges[seq_id], CONFIG['kmer_size'])
        relaxed_overlaps = [
            ov
            for ov in relaxed_overlaps
            if not (
                ov["range"][0] > ov["pos"][1] - CONFIG["min_frag_length"]
                or ov["range"][1] < ov["pos"][0] + CONFIG["min_frag_length"]
            )
        ]
        print(f"  Found {len(relaxed_overlaps)} relaxed overlaps")
        primer_flanked_overlaps[seq_id].extend(relaxed_overlaps)

attempting to bridge gaps in Maizel_COS-AT1G27430-GYF2
  Found 199 relaxed overlaps
attempting to bridge gaps in Maizel_COS-SETH5
  Found 45 relaxed overlaps
attempting to bridge gaps in Pereira_COS-SynDNA-f1
  Found 302 relaxed overlaps
attempting to bridge gaps in Pereira_COS-SynDNA-f2
  Found 39 relaxed overlaps


## Segment overlaps

They are suppused to sit in cur_seq_regrions (so to be homology free relative to the background and the current sequence)

In [28]:
segment_overlaps = {}
# 'Free end version' of overlaps, no boundary constraints
# for seq in sequences:
#     print(seq.id)
#     cur_overlaps = []
#     for region in cur_seq_regions[seq.id]:
#         if region[1] - region[0] < CONFIG['max_overlap'] and region[1] - region[0] >= CONFIG['min_overlap']:
#             cur_overlaps.append(region)
#             continue
#         for start in range(region[0], region[1] - CONFIG['max_overlap'] + 1, CONFIG['min_step']):
#             cur_overlaps.append((start, start + CONFIG['max_overlap']))
#         for end in range(region[1], region[0] + CONFIG['max_overlap'] - 1, -CONFIG['min_step']):
#             cur_overlaps.append((end - CONFIG['max_overlap'], end))
#     segment_overlaps[seq.id] = sorted(cur_overlaps, key=lambda x: x[0])

# And this one is for the boundary motif constraints It ignores the min_step
for seq in sequences:
    print(seq.id)
    seq_str = str(seq.seq).upper()
    cur_overlaps = []
    for region in cur_seq_regions[seq.id]:
        for m in re.finditer(CONFIG['seg_boundary_pattern'], seq_str[region[0]:region[1]]):
            left_boundary = region[0] + m.start()
            furthest_allowed = min(region[1], left_boundary + CONFIG['max_overlap'])
            for m2 in re.finditer(CONFIG['seg_boundary_pattern'], seq_str[left_boundary + CONFIG['min_overlap'] - 1:furthest_allowed]):
                right_boundary = left_boundary + CONFIG['min_overlap'] - 1 + m2.end()
                cur_overlaps.append((left_boundary, right_boundary))
    segment_overlaps[seq.id] = sorted(cur_overlaps, key=lambda x: x[0])

Maizel_COS-AT1G27430-GYF2
Maizel_COS-SETH5
Pereira_COS-SynDNA-f1
Pereira_COS-SynDNA-f2
PV252688r_p6utr
MK050105r_p6utr
MN450855r_p6utr
PQ488560r_p6utr
LC177792r_p6utr
AB890001r_p6utr
MH184583_rep
MG020022r_naive
OR500095r_p6utr
MN450853r_p6utr
JN998607r_p6utr
PQ537341r_p6utr
MZ289137_rep
MZ542728r_p6utr
PQ541186r_p6utr
ON644869r_p6utr
MG020022r_p6utr
MT840363_rep
OP610066r_p6utr
OR500095r_naive
PX021458r_p6utr
PQ488556r_p6utr


In [29]:
for seq_id in segment_overlaps:
    print(f"\n{seq_id}: {len(segment_overlaps[seq_id])} segment overlaps found")



Maizel_COS-AT1G27430-GYF2: 116 segment overlaps found

Maizel_COS-SETH5: 95 segment overlaps found

Pereira_COS-SynDNA-f1: 530 segment overlaps found

Pereira_COS-SynDNA-f2: 880 segment overlaps found

PV252688r_p6utr: 1549 segment overlaps found

MK050105r_p6utr: 1485 segment overlaps found

MN450855r_p6utr: 1474 segment overlaps found

PQ488560r_p6utr: 1526 segment overlaps found

LC177792r_p6utr: 770 segment overlaps found

AB890001r_p6utr: 1139 segment overlaps found

MH184583_rep: 1181 segment overlaps found

MG020022r_naive: 1068 segment overlaps found

OR500095r_p6utr: 1406 segment overlaps found

MN450853r_p6utr: 1297 segment overlaps found

JN998607r_p6utr: 1275 segment overlaps found

PQ537341r_p6utr: 1570 segment overlaps found

MZ289137_rep: 1221 segment overlaps found

MZ542728r_p6utr: 1410 segment overlaps found

PQ541186r_p6utr: 1350 segment overlaps found

ON644869r_p6utr: 1174 segment overlaps found

MG020022r_p6utr: 1086 segment overlaps found

MT840363_rep: 944 segm

In [30]:
# seg_overlap_gap_threshold = 1.5 * CONFIG['max_frag_length']
# seg_overlap_gaps = {}

# for seq in sequences:
#     seq_id = seq.id
#     seq_len = len(seq.seq)
#     ovs = sorted(segment_overlaps[seq_id], key=lambda x: x[0])
#     seg_overlap_gaps[seq_id] = []

#     if ovs:
#         if ovs[0][0] > seg_overlap_gap_threshold:
#             seg_overlap_gaps[seq_id].append((0, ovs[0][0]))
#         for i in range(len(ovs) - 1):
#             gap_start = ovs[i][1]
#             gap_end = ovs[i + 1][0]
#             if gap_end - gap_start > seg_overlap_gap_threshold:
#                 seg_overlap_gaps[seq_id].append((gap_start, gap_end))
#         if ovs[-1][1] < seq_len - seg_overlap_gap_threshold:
#             seg_overlap_gaps[seq_id].append((ovs[-1][1], seq_len))
#     else:
#         seg_overlap_gaps[seq_id].append((0, seq_len))

# for seq_id, g in seg_overlap_gaps.items():
#     if g:
#         print(f"\n{seq_id}: {len(g)} gap(s) in segment overlaps")
#         for gap_start, gap_end in g:
#             print(f"  {gap_start}-{gap_end} (length: {gap_end - gap_start})")


# for seq_id in seg_overlap_gaps:
#     if len(seg_overlap_gaps[seq_id]) > 0:
#         print(f"attempting to bridge gaps in segment overlaps for {seq_id}")
#         regions = cp.get_region_intersects(seg_overlap_gaps[seq_id], cur_seq_regions[seq_id], threshold=40)
#         seq_str = str(seq.seq).upper()
#         cur_overlaps = []
#         for region in regions:
#             for m in re.finditer(CONFIG['seg_boundary_pattern'], seq_str[region[0]:region[1]]):
#                 left_boundary = region[0] + m.start()
#                 furthest_allowed = min(region[1], left_boundary + CONFIG['max_overlap'])
#                 for m2 in re.finditer(CONFIG['seg_boundary_pattern'], seq_str[left_boundary + CONFIG['min_overlap'] - 16:furthest_allowed]):
#                     right_boundary = left_boundary + CONFIG['min_overlap'] - 16 + m2.end()
#                     cur_overlaps.append((left_boundary, right_boundary))
#         segment_overlaps[seq.id].extend(sorted(cur_overlaps, key=lambda x: x[0]))
#         print(f"  Found {len(cur_overlaps)} additional segment overlaps") 

## Shortest path algorithm

In [34]:
# This function defines an extra cost for sub-optimal edges. Generally this should be in range of 0 - 0.5 of the extra edge cost
# Can be larger for the too long primers to avoid them, but sometimes they are required
def get_edge_penalty(left_ov, right_ov, 
                     opt_primer_len, max_primer_len_diff, 
                     opt_overlap_len, max_overlap_len_diff, 
                     max_tm_diff=5.0):
    penalty = 0.0
    if left_ov['type'] == 'fr' and right_ov['type'] == 'fr':
        tm_diff = abs(left_ov['tm_left'] - right_ov['tm_right'])
        penalty += tm_diff / max_tm_diff * 0.1
    
    if left_ov['type'] == 'fr':
        primer_len_diff = abs(left_ov['pr_left_len'] - opt_primer_len)
        penalty += primer_len_diff / max_primer_len_diff * 0.1
    if right_ov['type'] == 'fr':
        primer_len_diff = abs(right_ov['pr_right_len'] - opt_primer_len)
        penalty += primer_len_diff / max_primer_len_diff * 0.1
    
    left_overlap_len = left_ov['pos'][1] - left_ov['pos'][0]
    right_overlap_len = right_ov['pos'][1] - right_ov['pos'][0]

    if left_overlap_len == 0:
        left_overlap_len = opt_overlap_len
    if right_overlap_len == 0:
        right_overlap_len = opt_overlap_len

    penalty += abs(left_overlap_len - opt_overlap_len) / max_overlap_len_diff * 0.1
    penalty += abs(right_overlap_len - opt_overlap_len) / max_overlap_len_diff * 0.1

    return round(penalty * 100)

# Check for heterodimers in primers
@lru_cache(maxsize=None)
def _heterodimer_tm(a, b):
    # Order-independent cache key so (a,b) and (b,a) share an entry.
    if a > b:
        a, b = b, a
    return primer3.calc_heterodimer_tm(a, b)

def is_heterodimer_risk(fwd, rev, cfg):
    """True if fwd/rev form a dimer hot enough to matter.

    An *extendable* primer-dimer needs the 3' end of one primer to anneal
    (antiparallel) to the other, i.e. the reverse-complement of its 3' anchor
    must occur as a substring of the partner. This is the same screen used for
    off-target priming, just with the partner primer in place of the template.
    primer3 is only consulted for pairs that pass this gate.
    """
    k = cfg['primer_3prime_anchor']
    suspect = (reverse_complement(fwd[-k:]) in rev or
                reverse_complement(rev[-k:]) in fwd)
    if not suspect:
        return False
    return _heterodimer_tm(fwd, rev) > cfg['max_heterodimer_tm']


# This function puts together all overlaps and adds the start and end dummy overlaps

def _has_boundary_motif(pos, seq, boundary_motif_pattern):
    start, end = pos
    starts_with_motif = boundary_motif_pattern.match(seq, start) is not None
    ends_with_motif = any(m.end() == end for m in boundary_motif_pattern.finditer(seq, 0, end))
    return starts_with_motif and ends_with_motif

def get_all_overlaps(primer_flanked_overlaps, segment_overlaps, seq_len, seq, boundary_motif_pattern):
    all_overlaps = (
        [
            {
                'pos': ov,
                'type': 'seg',
                'range': (0, seq_len),
                'extra_weight': 0.0
            }
            for ov in segment_overlaps
        ] +
        [
            {
                'pos': ov['pos'], 
                'type': 'fr', 
                'pr_left_len': len(ov['forward']['seq']),
                'pr_right_len': len(ov['reverse']['seq']),
                'pr_left_seq': ov['forward']['seq'],
                'pr_right_seq': ov['reverse']['seq'],
                'tm_left': ov['forward']['tm'],
                'tm_right': ov['reverse']['tm'],
                'range': ov['range'],
                'boundary_motif': _has_boundary_motif(ov['pos'], seq, boundary_motif_pattern),
                'extra_weight': ov.get('extra_weight', 0.0)
            } 
            for i, ov in enumerate(primer_flanked_overlaps)
        ] +
        [
            {
                'pos': (0, 0),
                'type': 'seg',
                'range': (0, seq_len),
                'extra_weight': 0.0
            }, 
            {
                'pos': (seq_len, seq_len),
                'type': 'seg',
                'range': (0, seq_len),
                'extra_weight': 0.0
            }
        ]
    )
    return sorted(all_overlaps, key=lambda x: x['pos'][0])

In [35]:
# The search algorithm itself
def get_overlap_states(all_overlaps, seq_len, cfg):

    min_length = cfg['min_frag_length']
    max_length = cfg['max_frag_length']
    min_seg_length = cfg['min_segment_length']
    opt_seg_length = cfg['opt_segment_length']
    segment_offset_left = cfg['segment_offset_left']
    segment_offset_right = cfg['segment_offset_right']
    max_overlap = cfg['max_overlap']
    min_overlap = cfg['min_overlap']
    seq_offset_left = cfg['seq_offset_left']
    seq_offset_right = cfg['seq_offset_right']
    opt_primer_len = cfg['opt_primer_len']
    max_primer_len_diff = cfg['max_primer_len_diff']

    overlap_states = [{} for _ in range(len(all_overlaps))]

    overlap_states[0][(0, seq_len)] = (0, -1, None) # state: (start_seg, end_range): (weight, predecessor_ind, predecessor_state)

    for i, ov in enumerate(all_overlaps):

        if i % 1000 == 0:
            print(f"Processing overlap {i}/{len(all_overlaps)}")
        if len(overlap_states[i]) == 0:
            continue
        j = i + 1
        too_far = False
        while j < len(all_overlaps) and not too_far:
            next_ov = all_overlaps[j]
            if next_ov['pos'][0] - max(ov['pos'][0], 0) > max_length:
                too_far = True
                continue

            length = next_ov['pos'][1] - max(ov['pos'][0], 0)
            if min_length <= length <= max_length:
                edge_weight = 100 + get_edge_penalty(ov, next_ov, 
                                                    opt_primer_len, max_primer_len_diff, 
                                                    max_overlap, max_overlap - min_overlap)
                # split in two: check if next_ov starts a new segment or if it continues the current one
                # segment start loop
                if next_ov['type'] == 'seg' or (next_ov['range'] == (0, seq_len) and next_ov['boundary_motif']):
                    for state, (cur_weight, _, _) in overlap_states[i].items():
                        if next_ov['range'][0] > state[0] or next_ov['pos'][1] > state[1]:
                            continue
                        seg_len = next_ov['pos'][1] - state[0]
                        if seg_len < min_seg_length:
                            continue

                        #segments must have extra space for flanking stuff
                        extra_length = segment_offset_right
                        if state[0] == ov['pos'][0]:
                            extra_length += segment_offset_left
                        if ov['pos'] == (0, 0):
                            extra_length += seq_offset_left
                        if next_ov['pos'] == (seq_len, seq_len):
                            extra_length += seq_offset_right
                        if length + extra_length > max_length:
                            continue

                        extra_penalty = max(opt_seg_length - seg_len, 0) / opt_seg_length * 500

                        new_weight = cur_weight + edge_weight + extra_penalty + next_ov['extra_weight']
                        new_state = (next_ov['pos'][0], next_ov['range'][1])

                        existing = overlap_states[j].get(new_state)
                        if existing is None or new_weight < existing[0]:
                            overlap_states[j][new_state] = (new_weight, i, state)

                # continue segment loop
                if next_ov['type'] == 'fr':
                    for state, (cur_weight, _, _) in overlap_states[i].items():
                        if next_ov['range'][0] > state[0] or next_ov['pos'][1] > state[1]:
                            continue
                        if next_ov['pos'][1] - state[0] > opt_seg_length:
                            # in this case only start segment option is possible
                            continue
                        extra_length = 0
                        if ov['pos'] == (0, 0):
                            extra_length += seq_offset_left
                        if next_ov['pos'] == (seq_len, seq_len):
                            extra_length += seq_offset_right
                        if state[0] == ov['pos'][0]:
                            extra_length += segment_offset_left
                        
                        if length + extra_length > max_length:
                            # if we are at the start of the segment, we need extra space on the left
                            continue
                        # This may take forever...
                        
                        if 'pr_left_seq' in ov and 'pr_right_seq' in next_ov and is_heterodimer_risk(ov['pr_left_seq'], next_ov['pr_right_seq'], cfg):
                            continue
                        new_weight = cur_weight + edge_weight + next_ov['extra_weight']
                        new_state = (state[0], min(next_ov['range'][1], state[1]))

                        existing = overlap_states[j].get(new_state)
                        if existing is None or new_weight < existing[0]:
                            overlap_states[j][new_state] = (new_weight, i, state)
                    
            j += 1
    return overlap_states

Run the search **(takes 1.5 hours on my laptop)**

In [36]:
full_overlap_states = {}
full_all_overlaps  = {}
for seq in sequences:
    print(seq.id)
    full_all_overlaps[seq.id] = get_all_overlaps(primer_flanked_overlaps[seq.id], 
                                                 segment_overlaps[seq.id], 
                                                 len(str(seq.seq)),
                                                 str(seq.seq),
                                                 CONFIG['seg_boundary_pattern'])

    full_overlap_states[seq.id] = get_overlap_states(full_all_overlaps[seq.id], len(str(seq.seq)), CONFIG)

# store results to file
state_file = Path("data/full_overlap_states.pkl")
with state_file.open("wb") as f:
    pickle.dump(full_overlap_states, f)
overlap_file = Path("data/full_all_overlaps.pkl")
with overlap_file.open("wb") as f:
    pickle.dump(full_all_overlaps, f)


Maizel_COS-AT1G27430-GYF2
Processing overlap 0/4730
Processing overlap 1000/4730
Processing overlap 2000/4730
Processing overlap 3000/4730
Processing overlap 4000/4730
Maizel_COS-SETH5
Processing overlap 0/5680
Processing overlap 1000/5680
Processing overlap 2000/5680
Processing overlap 3000/5680
Processing overlap 4000/5680
Processing overlap 5000/5680
Pereira_COS-SynDNA-f1
Processing overlap 0/11854
Processing overlap 1000/11854
Processing overlap 2000/11854
Processing overlap 3000/11854
Processing overlap 4000/11854
Processing overlap 5000/11854
Processing overlap 6000/11854
Processing overlap 7000/11854
Processing overlap 8000/11854
Processing overlap 9000/11854
Processing overlap 10000/11854
Processing overlap 11000/11854
Pereira_COS-SynDNA-f2
Processing overlap 0/15404
Processing overlap 1000/15404
Processing overlap 2000/15404
Processing overlap 3000/15404
Processing overlap 4000/15404
Processing overlap 5000/15404
Processing overlap 6000/15404
Processing overlap 7000/15404
Proc

### Update

Since f2 is sooo complicated, I also add some relaxed overlaps outside of the gap regions. It fails to find a path otherwise. It is probably an overshoot, but I'm really tired to try and fix this by adding only the required relaxations.

In [46]:
seq_id = "Pereira_COS-SynDNA-f2"

regions = cp.get_region_intersects([(14000, 18000)], bg_regions[seq_id], threshold=40)
relaxed_overlaps = get_relaxed_overlaps(seq_id, regions, anchor_kmers, sequences_by_id, CONFIG, extra_weight=500.0, max_tm=67.0, min_tm=57.0, min_overlap=40)
        
for ov in relaxed_overlaps:
    ov["range"] = get_overlap_range(ov["pos"][0], ov["pos"][1], homfree_ranges[seq_id], CONFIG['kmer_size'])
relaxed_overlaps = [
    ov
    for ov in relaxed_overlaps
    if not (
        ov["range"][0] > ov["pos"][1] - CONFIG["min_frag_length"]
        or ov["range"][1] < ov["pos"][0] + CONFIG["min_frag_length"]
    )
]
print(f"  Found {len(relaxed_overlaps)} relaxed overlaps")
primer_flanked_overlaps[seq_id].extend(relaxed_overlaps)

seq = sequences_by_id[seq_id]

full_all_overlaps[seq.id] = get_all_overlaps(primer_flanked_overlaps[seq.id], 
                                                segment_overlaps[seq.id], 
                                                len(str(seq.seq)),
                                                str(seq.seq),
                                                CONFIG['seg_boundary_pattern'])

full_overlap_states[seq.id] = get_overlap_states(full_all_overlaps[seq.id], len(str(seq.seq)), CONFIG)


  Found 5364 relaxed overlaps
Processing overlap 0/20768
Processing overlap 1000/20768
Processing overlap 2000/20768
Processing overlap 3000/20768
Processing overlap 4000/20768
Processing overlap 5000/20768
Processing overlap 6000/20768
Processing overlap 7000/20768
Processing overlap 8000/20768
Processing overlap 9000/20768
Processing overlap 10000/20768
Processing overlap 11000/20768
Processing overlap 12000/20768
Processing overlap 13000/20768
Processing overlap 14000/20768
Processing overlap 15000/20768
Processing overlap 16000/20768
Processing overlap 17000/20768
Processing overlap 18000/20768
Processing overlap 19000/20768
Processing overlap 20000/20768


In [48]:
state_file = Path("data/full_overlap_states.pkl")
with state_file.open("wb") as f:
    pickle.dump(full_overlap_states, f)
overlap_file = Path("data/full_all_overlaps.pkl")
with overlap_file.open("wb") as f:
    pickle.dump(full_all_overlaps, f)

## Continue

In [14]:
# read previously stored results from file
state_file = Path("data/full_overlap_states.pkl")
with state_file.open("rb") as f:
    full_overlap_states = pickle.load(f)
overlap_file = Path("data/full_all_overlaps.pkl")
with overlap_file.open("rb") as f:
    full_all_overlaps = pickle.load(f)

Checking if in all the sequences the end was reached.

In [47]:
for seq_id in full_overlap_states:
    print(seq_id)
    print(full_overlap_states[seq_id][-1])

Maizel_COS-AT1G27430-GYF2
{(5925, 5925): (1453.0, 4677, (2553, 5925))}
Maizel_COS-SETH5
{(8509, 8509): (1781.3, 5642, (4462, 8509))}
Pereira_COS-SynDNA-f1
{(21138, 21138): (3544.0, 11598, (15965, 21138))}
Pereira_COS-SynDNA-f2
{(25305, 25305): (5467.099999999999, 20507, (23016, 25305))}
PV252688r_p6utr
{(8124, 8124): (1281.6, 7489, (3106, 8124))}
MK050105r_p6utr
{(8131, 8131): (1283.8, 10258, (5009, 8131))}
MN450855r_p6utr
{(8112, 8112): (1289.2, 6885, (4027, 8112))}
PQ488560r_p6utr
{(8109, 8109): (1242.5, 8263, (4750, 8109))}
LC177792r_p6utr
{(7979, 7979): (1300.1, 5629, (3992, 7979))}
AB890001r_p6utr
{(7982, 7982): (1287.8, 7625, (4867, 7982))}
MH184583_rep
{(7452, 7452): (1304.7, 9660, (4479, 7452))}
MG020022r_naive
{(8059, 8059): (1365.3, 7818, (3304, 8059))}
OR500095r_p6utr
{(8124, 8124): (1275.1, 7432, (3904, 8124))}
MN450853r_p6utr
{(8155, 8155): (1291.6, 6068, (3185, 8155))}
JN998607r_p6utr
{(8004, 8004): (1250.6, 8724, (3935, 8004))}
PQ537341r_p6utr
{(8134, 8134): (1280.800000

In [49]:
def reconstruct_shortest_path(overlap_states, end_index=None):
    path = []
    current_index = len(overlap_states) - 1 if end_index is None else end_index
    current_state = min(overlap_states[current_index].items(), key=lambda x: x[1][0])[0]

    while current_index != -1:
        path.append((current_index, current_state))
        _, predecessor_index, predecessor_state = overlap_states[current_index][current_state]
        current_index = predecessor_index
        current_state = predecessor_state

    return path[::-1]

## Extract segments and fragments from the shortest path

In [50]:
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
import os


def _is_virtual_sentinel(ov, seq_len):
    """True for the two dummy bookend overlaps added in get_all_overlaps."""
    return ov['pos'][1] <= 0 or ov['pos'][0] >= seq_len


def get_end_index(states):
    """(end_index, is_partial). end_index None + is_partial False -> full path.
    end_index None + is_partial True -> no solution at all."""
    if states[-1]:
        return None, False
    for idx in range(len(states) - 1, -1, -1):
        if states[idx]:
            return idx, True
    return None, True


def extract_path_structure(seq_id, states, all_overlaps, seq_len, end_index=None):
    """Split the shortest path into segments and their fragments.

    Returns (path, segments); each segment is a dict with
      'nodes'  : the path nodes of the segment (shared boundary node at each end),
      'frags'  : [(start, end), ...] fragment spans in insert coordinates,
      'seg_start'/'seg_end' : segment extent in insert coordinates.
    """
    path = reconstruct_shortest_path(states, end_index)

    boundary = [i for i, (ov_idx, state) in enumerate(path)
                if state[0] == all_overlaps[ov_idx]['pos'][0]]
    # a partial path can finish mid-segment; close it on its last node
    if path and path[-1][1][0] != all_overlaps[path[-1][0]]['pos'][0]:
        boundary.append(len(path) - 1)

    segments = []
    for si in range(len(boundary) - 1):
        seg_nodes = path[boundary[si]:boundary[si + 1] + 1]
        left_ov = all_overlaps[seg_nodes[0][0]]
        right_ov = all_overlaps[seg_nodes[-1][0]]
        if _is_virtual_sentinel(left_ov, seq_len) and _is_virtual_sentinel(right_ov, seq_len):
            continue
        frags = []
        for j in range(1, len(seg_nodes)):
            a = max(all_overlaps[seg_nodes[j - 1][0]]['pos'][0], 0)
            b = min(all_overlaps[seg_nodes[j][0]]['pos'][1], seq_len)
            if a < b:
                frags.append((a, b))
        if not frags:
            continue
        segments.append({'nodes': seg_nodes, 'frags': frags,
                         'seg_start': frags[0][0], 'seg_end': frags[-1][1]})
    return path, segments

In [51]:
def build_annotated_record(seq_id, path, segments, all_overlaps):
    """Whole-sequence GenBank record annotated with non-homologous regions,
    the overlaps on the path, segments, fragments and primer-binding sites."""
    seq_record = sequences_by_id[seq_id]
    seq_len = len(seq_record.seq)
    rec = SeqRecord(seq_record.seq, id=seq_id[:16], name=seq_id[:16],
                    description=seq_id, annotations={"molecule_type": "DNA"})
    features = []

    for start, end in cur_seq_regions.get(seq_id, []):
        features.append(SeqFeature(FeatureLocation(start, end), type="misc_feature",
                        qualifiers={"label": ["non_homologous"]}))

    for ov_idx, state in path:
        ov = all_overlaps[ov_idx]
        if _is_virtual_sentinel(ov, seq_len):
            continue
        prefix = "junc" if state[0] == ov['pos'][0] else "ov"
        features.append(SeqFeature(FeatureLocation(ov['pos'][0], ov['pos'][1]),
                        type="repeat_region",
                        qualifiers={"label": [f"{prefix}_{ov['pos'][0]}_{ov['pos'][1]}"],
                                    "note": [f"type={ov['type']}"]}))

    pr_by_pos = {tuple(ov['pos']): ov for ov in primer_flanked_overlaps.get(seq_id, [])}
    for k, seg in enumerate(segments, start=1):
        features.append(SeqFeature(FeatureLocation(seg['seg_start'], seg['seg_end']),
                        type="gene",
                        qualifiers={"label": [f"Segment_{k}"],
                                    "note": [f"{len(seg['frags'])} fragment(s)"]}))
        for fi, (a, b) in enumerate(seg['frags'], start=1):
            features.append(SeqFeature(FeatureLocation(a, b), type="misc_feature",
                            qualifiers={"label": [f"S{k}_F{fi}"]}))
        for ov_idx, _state in seg['nodes'][1:-1]:
            ov = all_overlaps[ov_idx]
            entry = pr_by_pos.get(tuple(ov['pos']))
            if ov['type'] != 'fr' or entry is None:
                continue
            fwd, rev = entry['forward'], entry['reverse']
            features.append(SeqFeature(FeatureLocation(fwd['pos'][0], fwd['pos'][1], strand=1),
                            type="primer_bind",
                            qualifiers={"label": [f"S{k}_F"], "note": [f"Tm={fwd['tm']:.1f}"],
                                        "sequence": [fwd['seq']]}))
            features.append(SeqFeature(FeatureLocation(rev['pos'][0], rev['pos'][1], strand=-1),
                            type="primer_bind",
                            qualifiers={"label": [f"S{k}_R"], "note": [f"Tm={rev['tm']:.1f}"],
                                        "sequence": [rev['seq']]}))

    rec.features = sorted(features, key=lambda f: int(f.location.start))
    return rec


path_structures = {}
os.makedirs("annotated", exist_ok=True)

for seq_id in full_overlap_states:
    states = full_overlap_states[seq_id]
    all_overlaps = full_all_overlaps[seq_id]
    seq_len = len(sequences_by_id[seq_id].seq)

    if len(states) != len(all_overlaps):
        print(f"{seq_id}: SKIP - saved states/all_overlaps out of sync "
              f"({len(states)} vs {len(all_overlaps)}); re-run the search cell for it")
        continue

    end_index, partial = get_end_index(states)
    if end_index is None and partial:
        print(f"{seq_id}: no solution found, skipping")
        continue

    path, segments = extract_path_structure(seq_id, states, all_overlaps, seq_len, end_index)
    path_structures[seq_id] = {'path': path, 'segments': segments, 'partial': partial}
    suffix = "_partial" if partial else ""

    record = build_annotated_record(seq_id, path, segments, all_overlaps)
    with open(f"annotated/{seq_id}{suffix}.gb", "w") as fh:
        SeqIO.write(record, fh, "genbank")

    frag_recs, n = [], 0
    for seg in segments:
        for (a, b) in seg['frags']:
            n += 1
            frag_recs.append(SeqRecord(sequences_by_id[seq_id].seq[a:b],
                             id=f"{seq_id}_F{n}", name=f"{seq_id}_F{n}"[:16],
                             description=f"pos={a}-{b}"))
    with open(f"annotated/{seq_id}{suffix}_fragments.fa", "w") as fh:
        SeqIO.write(frag_recs, fh, "fasta")

    print(f"{seq_id}{suffix}: {len(segments)} segment(s), {n} fragment(s)")

Maizel_COS-AT1G27430-GYF2: 2 segment(s), 9 fragment(s)
Maizel_COS-SETH5: 2 segment(s), 12 fragment(s)
Pereira_COS-SynDNA-f1: 4 segment(s), 28 fragment(s)
Pereira_COS-SynDNA-f2: 7 segment(s), 34 fragment(s)
PV252688r_p6utr: 2 segment(s), 10 fragment(s)
MK050105r_p6utr: 2 segment(s), 10 fragment(s)
MN450855r_p6utr: 2 segment(s), 10 fragment(s)
PQ488560r_p6utr: 2 segment(s), 10 fragment(s)
LC177792r_p6utr: 2 segment(s), 10 fragment(s)
AB890001r_p6utr: 2 segment(s), 10 fragment(s)
MH184583_rep: 2 segment(s), 10 fragment(s)
MG020022r_naive: 2 segment(s), 11 fragment(s)
OR500095r_p6utr: 2 segment(s), 10 fragment(s)
MN450853r_p6utr: 2 segment(s), 10 fragment(s)
JN998607r_p6utr: 2 segment(s), 10 fragment(s)
PQ537341r_p6utr: 2 segment(s), 10 fragment(s)
MZ289137_rep: 2 segment(s), 9 fragment(s)
MZ542728r_p6utr: 2 segment(s), 10 fragment(s)
PQ541186r_p6utr: 2 segment(s), 10 fragment(s)
ON644869r_p6utr: 2 segment(s), 10 fragment(s)
MG020022r_p6utr: 2 segment(s), 11 fragment(s)
MT840363_rep: 2 seg

## Add the TT1 / TT2 backbone arms and NotI cut sites

In [52]:
synDNA_templ_TT1 = SeqIO.read("data/20260219-synDNA_template_4TAR_TT1.gb", "genbank").seq.upper()
synDNA_templ_TT2 = SeqIO.read("data/20260219-synDNA_template_4TAR_TT2.gb", "genbank").seq.upper()

PLACEHOLDER = "N" * 10 

_tt1_sep = synDNA_templ_TT1.find(PLACEHOLDER)
tt1_left_arm = synDNA_templ_TT1[:_tt1_sep]
tt1_right_arm = synDNA_templ_TT1[_tt1_sep + len(PLACEHOLDER):]

_tt2_sep = synDNA_templ_TT2.find(PLACEHOLDER)
tt2_left_arm = synDNA_templ_TT2[:_tt2_sep]
tt2_right_arm = synDNA_templ_TT2[_tt2_sep + len(PLACEHOLDER):]

NOTI = Seq("GCGGCCGC")  

def _add_noti_5(seq):
    return NOTI + seq[2:] if seq[:2] == Seq("GC") else NOTI + seq


def _add_noti_3(seq):
    return seq[:-2] + NOTI if seq[-2:] == Seq("GC") else seq + NOTI


print("TT1 arms:", len(tt1_left_arm), len(tt1_right_arm),
      "| TT2 arms:", len(tt2_left_arm), len(tt2_right_arm))

TT1 arms: 78 108 | TT2 arms: 126 105


In [53]:
def flank_segment_fragments(seq_id, segments):
    """List of segments, each a list of SeqRecords carrying the backbone edits:
    TT2 arms at the two insert extremities and NotI + TT1 arms at segment edges."""
    insert = sequences_by_id[seq_id].seq
    n_seg = len(segments)
    out = []
    for k, seg in enumerate(segments):
        frag_seqs = [insert[a:b] for (a, b) in seg['frags']]

        if k == 0:
            frag_seqs[0] = tt2_left_arm + frag_seqs[0]
        if k == n_seg - 1:
            frag_seqs[-1] = frag_seqs[-1] + tt2_right_arm

        # NotI sites + TT1 homology arms at the segment edges
        if len(frag_seqs) == 1:
            s = _add_noti_3(_add_noti_5(frag_seqs[0]))
            frag_seqs[0] = tt1_left_arm + s + tt1_right_arm
        else:
            frag_seqs[0] = tt1_left_arm + _add_noti_5(frag_seqs[0])
            frag_seqs[-1] = _add_noti_3(frag_seqs[-1]) + tt1_right_arm

        recs = [SeqRecord(sq, id=f"{seq_id}_TT1seg{k+1}_f{fi+1}",
                          name=f"seg{k+1}_f{fi+1}", description="")
                for fi, sq in enumerate(frag_seqs)]
        out.append(recs)
    return out

tt1_segment_dict = {}
for seq_id, ps in path_structures.items():
    tt1_segment_dict[seq_id] = flank_segment_fragments(seq_id, ps['segments'])
    n_frags = sum(len(s) for s in tt1_segment_dict[seq_id])
    max_len = max(len(r.seq) for s in tt1_segment_dict[seq_id] for r in s)
    print(f"{seq_id}: {len(tt1_segment_dict[seq_id])} segment(s), "
          f"{n_frags} fragment(s), longest flanked fragment {max_len} bp")

Maizel_COS-AT1G27430-GYF2: 2 segment(s), 9 fragment(s), longest flanked fragment 987 bp
Maizel_COS-SETH5: 2 segment(s), 12 fragment(s), longest flanked fragment 993 bp
Pereira_COS-SynDNA-f1: 4 segment(s), 28 fragment(s), longest flanked fragment 999 bp
Pereira_COS-SynDNA-f2: 7 segment(s), 34 fragment(s), longest flanked fragment 1000 bp
PV252688r_p6utr: 2 segment(s), 10 fragment(s), longest flanked fragment 1000 bp
MK050105r_p6utr: 2 segment(s), 10 fragment(s), longest flanked fragment 999 bp
MN450855r_p6utr: 2 segment(s), 10 fragment(s), longest flanked fragment 1000 bp
PQ488560r_p6utr: 2 segment(s), 10 fragment(s), longest flanked fragment 993 bp
LC177792r_p6utr: 2 segment(s), 10 fragment(s), longest flanked fragment 1000 bp
AB890001r_p6utr: 2 segment(s), 10 fragment(s), longest flanked fragment 998 bp
MH184583_rep: 2 segment(s), 10 fragment(s), longest flanked fragment 990 bp
MG020022r_naive: 2 segment(s), 11 fragment(s), longest flanked fragment 975 bp
OR500095r_p6utr: 2 segment(s)

## Write the flanked fragments to multi-fasta, one TT1 segment at a time

In [54]:
os.makedirs("Intermediates", exist_ok=True)
for seq_id, segments in tt1_segment_dict.items():
    for seg_idx, segment_frags in enumerate(segments, start=1):
        fname = f"Intermediates/{seq_id}_TT1seg{seg_idx}.fa"
        with open(fname, "w") as fao:
            SeqIO.write(segment_frags, fao, "fasta")
        print(f"Wrote {fname} ({len(segment_frags)} fragment(s))")

Wrote Intermediates/Maizel_COS-AT1G27430-GYF2_TT1seg1.fa (4 fragment(s))
Wrote Intermediates/Maizel_COS-AT1G27430-GYF2_TT1seg2.fa (5 fragment(s))
Wrote Intermediates/Maizel_COS-SETH5_TT1seg1.fa (6 fragment(s))
Wrote Intermediates/Maizel_COS-SETH5_TT1seg2.fa (6 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f1_TT1seg1.fa (7 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f1_TT1seg2.fa (7 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f1_TT1seg3.fa (7 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f1_TT1seg4.fa (7 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f2_TT1seg1.fa (6 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f2_TT1seg2.fa (5 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f2_TT1seg3.fa (6 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f2_TT1seg4.fa (3 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f2_TT1seg5.fa (4 fragment(s))
Wrote Intermediates/Pereira_COS-SynDNA-f2_TT1seg6.fa (6 fragment(s))
Wrote Intermediates/Pereira_COS-SynD

## Testing the cloning strategy

Same assembly test as the TAR-assembly notebook, in two hierarchical Gibson/TAR
steps:

1. **TT1 assembly** – each segment's fragments are assembled together with the
   TT1 hooks and the BsaI-linearised TT1 vector into a TT1 plasmid.
2. **TT2 assembly** – each TT1 plasmid is NotI-digested to release its segment,
   and all segments are assembled with the TT2 hooks and the BsaI-linearised TT2
   vector into the final construct.

> Requires `pydna` and the backbone/hook GenBank files (`pSDL42-TT1`,
> `pSDL77-TT2-TAR_shuffle-pCC1`, `pSDL36-TT1L`, `pSDL37-TT1R`, `pSDL38-TT2L`,
> `pSDL39-TT2R`) placed in `RESOURCE_DIR` below.

In [ ]:
from pydna.parsers import parse
from pydna.amplify import pcr
from pydna.assembly2 import gibson_assembly  # tolerates terminal 5' overhang mismatches
from Bio.Restriction import BsaI, NotI
import glob

# Drop the TAR pSDL* GenBank files here to run the assembly test.
RESOURCE_DIR = Path("../DNA_resources")

# Vectors, linearised with BsaI for Gibson/TAR
TT1_vector = parse(str(RESOURCE_DIR / "pSDL42-TT1.gb"))[0]                    # high copy TT1
_dropout, TT1_vector_lin = TT1_vector.cut(BsaI)
TT2_vector = parse(str(RESOURCE_DIR / "pSDL77-TT2-TAR_shuffle-pCC1.gb"))[0]  # low copy TT2
_dropout, TT2_vector_lin = TT2_vector.cut(BsaI)

# TAR hooks, amplified from their templates
TT1L_hook = pcr("GCGCGCTCACTGGCCGTCG", "ttcgtcgtccgattcgtc",
                parse(str(RESOURCE_DIR / "pSDL36-TT1L.gb"))[0].seq, limit=17)
TT1R_hook = pcr("GGGTTAATTGCGCGCTTGG", "GTTGTGAGTCAAGATGTCGTTGGC",
                parse(str(RESOURCE_DIR / "pSDL37-TT1R.gb"))[0].seq, limit=17)
TT2L_hook = pcr("CCGGGCCTTTCTTTATGTTTTTG", "cagcgatcgcatccatggc",
                parse(str(RESOURCE_DIR / "pSDL38-TT2L.gb"))[0].seq, limit=17)
TT2R_hook = pcr("CTGGCAAGCCACGTTTGG", "gttgtgtgttaggatgtcgttcg",
                parse(str(RESOURCE_DIR / "pSDL39-TT2R.gb"))[0].seq, limit=17)

In [ ]:
# Load the flanked fragments back, one TT1 segment at a time
fragments_segments = {}
for fname in sorted(glob.glob("Intermediates/*_TT1seg*.fa")):
    basename = os.path.basename(fname).replace(".fa", "")
    seq_id, seg_idx = basename.split("_TT1seg")
    seg_idx = int(seg_idx)
    fragments_segments.setdefault(seq_id, {})[seg_idx] = parse(fname)
    print(f"Loaded {seq_id} segment {seg_idx}: "
          f"{len(fragments_segments[seq_id][seg_idx])} fragment(s)")

### TT1 assembly (segments)

In [ ]:
os.makedirs("Intended_designs", exist_ok=True)
assembled_segments = {}

for seq_id, segments_dict in fragments_segments.items():
    assembled_segments[seq_id] = {}
    for seg_idx, fragments in segments_dict.items():
        asm = gibson_assembly(fragments + [TT1L_hook, TT1_vector_lin, TT1R_hook])
        product = asm[0]  # the other product is just the reverse complement
        product.annotations["molecule_type"] = "DNA"
        fname = f"Intended_designs/{seq_id}_TT1seg{seg_idx}_assembly.gb"
        with open(fname, "w") as gbo:
            SeqIO.write(product, gbo, "genbank")
        assembled_segments[seq_id][seg_idx] = product
        print(f"Assembled {seq_id} segment {seg_idx}: wrote {fname}")

print(f"\nTotal segments assembled: {sum(len(d) for d in assembled_segments.values())}")

### TT2 assembly (NotI-release the segments, join into the full construct)

In [ ]:
assembled_constructs = {}

for seq_id, segments_dict in assembled_segments.items():
    cut_segments = []
    for k in sorted(segments_dict):
        cut_segment, _TT1_backbone = segments_dict[k].cut(NotI)
        cut_segment.features.append(SeqFeature(
            location=FeatureLocation(0, len(cut_segment.seq)),
            type="segment",
            qualifiers={"id": f"{seq_id}_TT1seg{k}", "note": f"Segment {k}"}))
        cut_segments.append(cut_segment)

    asm = gibson_assembly(cut_segments + [TT2L_hook, TT2_vector_lin, TT2R_hook])
    product = asm[0]  # the other product is just the reverse complement
    product.annotations["molecule_type"] = "DNA"
    fname = f"Intended_designs/{seq_id}_TT2_assembly.gb"
    with open(fname, "w") as gbo:
        SeqIO.write(product, gbo, "genbank")
    assembled_constructs[seq_id] = product
    print(f"Assembled {seq_id} from {len(cut_segments)} segment(s): wrote {fname}")

print(f"\nTotal constructs assembled: {len(assembled_constructs)}")

## Diagnostic function 

In [ ]:
def diagnose_edge(seq_id, i, j, all_overlaps=None, overlap_states=None, cfg=None,
                  min_length=None, max_length=None, min_seg_length=None,
                  opt_seg_length=None, segment_offset_left=None,
                  segment_offset_right=None, max_overlap=None, min_overlap=None,
                  seq_offset_left=None, seq_offset_right=None):
    """Explain why the edge i -> j was (or was not) added in get_overlap_states.

    By default uses full_all_overlaps[seq_id], full_overlap_states[seq_id] and
    CONFIG values.  Override any parameter to test hypothetical scenarios.

    Returns a list of (state, [reason_strings]) pairs for each rejected state,
    or an empty list if the edge would be accepted for at least one state.
    """
    if all_overlaps is None:
        all_overlaps = full_all_overlaps[seq_id]
    if overlap_states is None:
        overlap_states = full_overlap_states[seq_id]
    if cfg is None:
        cfg = CONFIG

    if min_length is None:         min_length         = cfg['min_frag_length']
    if max_length is None:         max_length         = cfg['max_frag_length']
    if min_seg_length is None:     min_seg_length     = cfg['min_segment_length']
    if opt_seg_length is None:     opt_seg_length     = cfg['opt_segment_length']
    if segment_offset_left is None:  segment_offset_left  = cfg['segment_offset_left']
    if segment_offset_right is None: segment_offset_right = cfg['segment_offset_right']
    if max_overlap is None:        max_overlap        = cfg['max_overlap']
    if min_overlap is None:        min_overlap        = cfg['min_overlap']
    if seq_offset_left is None:    seq_offset_left    = cfg['seq_offset_left']
    if seq_offset_right is None:   seq_offset_right   = cfg['seq_offset_right']

    seq_len = len(sequences_by_id[seq_id].seq)
    ov      = all_overlaps[i]
    next_ov = all_overlaps[j]

    print(f"Diagnosing edge {i} -> {j}  [{seq_id}]")
    print(f"  ov[{i}]:       pos={ov['pos']}, type={ov['type']}, range={ov['range']}")
    print(f"  next_ov[{j}]:  pos={next_ov['pos']}, type={next_ov['type']}, range={next_ov['range']}")
    print()

    # ── pre-state checks ────────────────────────────────────────────────────────
    dist   = next_ov['pos'][0] - max(ov['pos'][0], 0)
    length = next_ov['pos'][1] - max(ov['pos'][0], 0)

    if dist > max_length:
        print(f"  [FAIL] too_far: gap {dist} > max_length {max_length}")
        return [("pre-state", [f"too_far: {dist} > {max_length}"])]

    if not (min_length <= length <= max_length):
        print(f"  [FAIL] length {length} not in [{min_length}, {max_length}]")
        return [("pre-state", [f"length {length} not in [{min_length}, {max_length}]"])]

    print(f"  length={length}  (OK: [{min_length}, {max_length}])")

    states_at_i = overlap_states[i]
    if not states_at_i:
        msg = f"no states reached at index {i} — overlap was never visited by the search"
        print(f"  [FAIL] {msg}")
        return [("pre-state", [msg])]

    print(f"  states at ov[{i}]: {len(states_at_i)}")

    # ── branch applicability ────────────────────────────────────────────────────
    is_seg_start = (next_ov['type'] == 'seg' or
                    (next_ov['range'] == (0, seq_len) and next_ov.get('boundary_motif', False)))
    is_fr_continue = (next_ov['type'] == 'fr')

    all_rejections = []

    # ── seg-start branch ────────────────────────────────────────────────────────
    if is_seg_start:
        print("\n  [seg-start branch]")
        for state, (cur_weight, _, _) in states_at_i.items():
            r = []
            if next_ov['range'][0] > state[0]:
                r.append(f"next_ov range[0] {next_ov['range'][0]} > state seg_start {state[0]}")
            if next_ov['pos'][1] > state[1]:
                r.append(f"next_ov pos[1] {next_ov['pos'][1]} > state end_range {state[1]}")
            seg_len = next_ov['pos'][1] - state[0]
            if seg_len < min_seg_length:
                r.append(f"seg_len {seg_len} < min_seg_length {min_seg_length}")
            extra = segment_offset_right
            if state[0] == ov['pos'][0]:   extra += segment_offset_left
            if ov['pos'] == (0, 0):         extra += seq_offset_left
            if next_ov['pos'] == (seq_len, seq_len): extra += seq_offset_right
            if length + extra > max_length:
                r.append(f"length {length} + extra_length {extra} = {length+extra} > max_length {max_length}")
            if r:
                print(f"    state={state} → REJECTED:")
                for reason in r: print(f"      - {reason}")
                all_rejections.append((state, r))
            else:
                print(f"    state={state} → ACCEPTED  (cur_weight={cur_weight})")
    else:
        print(f"\n  [seg-start branch] N/A (type={next_ov['type']}, boundary_motif={next_ov.get('boundary_motif')})")

    # ── fr-continue branch ──────────────────────────────────────────────────────
    if is_fr_continue:
        print("\n  [fr-continue branch]")
        for state, (cur_weight, _, _) in states_at_i.items():
            r = []
            if next_ov['range'][0] > state[0]:
                r.append(f"next_ov range[0] {next_ov['range'][0]} > state seg_start {state[0]}")
            if next_ov['pos'][1] > state[1]:
                r.append(f"next_ov pos[1] {next_ov['pos'][1]} > state end_range {state[1]}")
            cumlen = next_ov['pos'][1] - state[0]
            if cumlen > opt_seg_length:
                r.append(f"cumulative seg length {cumlen} > opt_seg_length {opt_seg_length}")
            extra = 0
            if ov['pos'] == (0, 0):         extra += seq_offset_left
            if next_ov['pos'] == (seq_len, seq_len): extra += seq_offset_right
            if state[0] == ov['pos'][0]:   extra += segment_offset_left
            if length + extra > max_length:
                r.append(f"length {length} + extra_length {extra} = {length+extra} > max_length {max_length}")
            if not r and 'pr_left_seq' in ov and 'pr_right_seq' in next_ov:
                if is_heterodimer_risk(ov['pr_left_seq'], next_ov['pr_right_seq'], cfg):
                    r.append(f"heterodimer risk: fwd={ov['pr_left_seq']}  rev={next_ov['pr_right_seq']}")
            if r:
                print(f"    state={state} → REJECTED:")
                for reason in r: print(f"      - {reason}")
                all_rejections.append((state, r))
            else:
                print(f"    state={state} → ACCEPTED  (cur_weight={cur_weight})")
    else:
        print(f"\n  [fr-continue branch] N/A (type={next_ov['type']})")

    if not all_rejections:
        print("\n  No rejection reasons found — edge should be present.")
        states_at_j = overlap_states[j]
        print(f"  overlap_states[{j}] has {len(states_at_j)} state(s): {list(states_at_j.keys())}")

    return all_rejections
